## Ngày 3 về RAG

### Chuyên gia hỏi đáp cho InsureLLM

Triển khai một pipeline RAG bằng LangChain 1.0.

Sử dụng VectorStore mà chúng ta đã tạo lần trước (với Hugging Face `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv  # Nhập load_dotenv từ gói dotenv.
from langchain_openai import ChatOpenAI  # Nhập ChatOpenAI từ gói langchain_openai.

from langchain_chroma import Chroma  # Nhập Chroma từ gói langchain_chroma.
from langchain_core.messages import SystemMessage, HumanMessage  # Nhập SystemMessage, HumanMessage từ gói langchain_core.messages.
from langchain_huggingface import HuggingFaceEmbeddings  # Nhập HuggingFaceEmbeddings từ gói langchain_huggingface.
import gradio as gr  # Nạp gradio as gr để sử dụng trong notebook.

In [2]:
MODEL = "gpt-4.1-nano"  # Chọn tên mô hình ngôn ngữ sẽ được sử dụng.
DB_NAME = "vector_db"  # Đặt tên thư mục cơ sở dữ liệu vectơ.
load_dotenv(override=True)  # Nạp lại các biến trong tệp `.env` vào môi trường chạy.

True

### Kết nối với Chroma; sử dụng Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # Khởi tạo mô hình dùng để tạo embedding cho văn bản.
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)  # Kết nối hoặc tạo kho vectơ Chroma.

### Thiết lập 2 đối tượng LangChain chính: retriever và llm

#### Giải thích thêm về "temperature":
- Kiểm soát mức độ đa dạng của đầu ra
- Temperature bằng 0 nghĩa là đầu ra sẽ có tính dự đoán được
- Temperature cao hơn giúp câu trả lời đa dạng hơn

Một số người mô tả temperature giống như "độ sáng tạo", nhưng cách hiểu đó chưa hoàn toàn chính xác.
- Thực chất, nó kiểm soát việc lựa chọn token trong quá trình suy luận
- temperature=0 nghĩa là: luôn chọn token có xác suất cao nhất
- temperature=1 thường nghĩa là: một token có xác suất 10% sẽ được chọn trong 10% số lần

Lưu ý: temperature bằng 0 không có nghĩa là đầu ra luôn có thể tái lập. Bạn cũng cần đặt một hạt giống ngẫu nhiên. Chúng ta sẽ thực hiện việc đó trong các tuần 6–8. (Ngay cả khi đó, kết quả cũng không phải lúc nào cũng tái lập được.)

Lưu ý 2: nếu muốn có tính sáng tạo, hãy sử dụng System Prompt!

In [4]:
retriever = vectorstore.as_retriever()  # Tạo bộ truy xuất từ kho vectơ.
llm = ChatOpenAI(temperature=0, model_name=MODEL)  # Khởi tạo mô hình trò chuyện với temperature bằng 0.

### Các đối tượng LangChain này triển khai phương thức `invoke()`

In [5]:
retriever.invoke("Avery là ai?")  # Thử bộ truy xuất với câu hỏi mẫu và hiển thị các tài liệu tìm được.

[Document(id='3acf6dd9-cb2b-49c8-bfa4-ecc95835a698', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2018**: **Exceeds Expectations**  \n  Under Avery’s pivoted vision, Insurellm launched two new successful products that significantly increased market share.  \n\n- **2019**: **Meets Expectations**  \n  Steady growth, however, some team tensions led to a minor drop in employee morale. Avery recognized the need to enhance company culture.  \n\n- **2020**: **Below Expectations**  \n  The COVID-19 pandemic posed unforeseen operational difficulties. Avery faced criticism for delayed strategy shifts, although efforts were eventually made to stabilize the company.  \n\n- **2021**: **Exceptional**  \n  Avery's decisive transition to remote work and rapid adoption of digital tools led to record-high customer satisfaction levels and increased sales.  \n\n- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics an

In [6]:
llm.invoke("Avery là ai?")  # Gửi trực tiếp câu hỏi mẫu tới LLM để so sánh với kết quả truy xuất.

AIMessage(content='Avery có thể là tên của nhiều người hoặc nhân vật khác nhau, tùy vào ngữ cảnh. Bạn có thể cung cấp thêm thông tin hoặc làm rõ hơn về Avery mà bạn muốn biết để mình có thể giúp đỡ chính xác hơn không?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 12, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_d0c31ec33f', 'id': 'chatcmpl-EObX5Nnd0yJnM9hL8QY6CutJHgsBy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--18d2c836-83e6-4bd1-8272-b0fae0ce4254-0', usage_metadata={'input_tokens': 12, 'output_tokens': 52, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

## Đã đến lúc ghép mọi thứ lại với nhau!

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
Bạn là một trợ lý am hiểu và thân thiện, đại diện cho công ty Insurellm.
Bạn đang trò chuyện với người dùng về Insurellm.
Nếu phù hợp, hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi.
Nếu không biết câu trả lời, hãy nói rõ điều đó.
Ngữ cảnh:
{context}
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `SYSTEM_PROMPT_TEMPLATE`.

In [10]:
def answer_question(question: str, history):  # Khai báo hàm trả lời câu hỏi bằng quy trình RAG.
    docs = retriever.invoke(question)  # Truy xuất các tài liệu liên quan đến câu hỏi.
    context = "\n\n".join(doc.page_content for doc in docs)  # Ghép nội dung các tài liệu truy xuất thành một chuỗi ngữ cảnh.
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)  # Tạo prompt hệ thống hoàn chỉnh cho lần gọi mô hình này.
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])  # Gọi mô hình và lưu phản hồi trả về.
    return response.content  # Trả kết quả này về cho nơi gọi hàm.

In [11]:
answer_question("Averi Lancaster là ai?", [])  # Chạy toàn bộ quy trình RAG cho câu hỏi mẫu và hiển thị câu trả lời.

NameError: name 'SYSTEM_PROMPT_TEMPLATE' is not defined

## Tiếp theo có thể là gì nhỉ? 😂

In [9]:
gr.ChatInterface(answer_question).launch()  # Khởi chạy giao diện Gradio để người dùng trò chuyện với hệ thống.

c:\Users\user\Desktop\llm_engineering_CuongPhan\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Thừa nhận đi — bạn đã nghĩ RAG sẽ phức tạp hơn thế này nhiều đúng không!!